# Step 7 — Proyek Mini: Membangun Mesin Pencari Utuh

**Tujuan praktikum**
1. Mahasiswa mampu mengintegrasikan preprocessing, indexing, model retrieval, dan evaluasi menjadi satu sistem.
2. Mahasiswa mampu melakukan eksperimen ablasi untuk melihat kontribusi tiap tahap preprocessing.
3. Mahasiswa mampu menyajikan hasil pencarian yang informatif (judul, skor, cuplikan).

In [1]:
# Jalankan sekali jika pustaka belum terpasang:
# !pip install pandas numpy matplotlib Sastrawi
import pandas as pd
pd.set_option("display.max_colwidth", 120)

In [2]:
import re
from stki_utils import (load_corpus, corpus_texts, preprocess, TfidfModel, BM25Model,
                        load_queries, load_qrels, evaluate)


class MesinPencari:
    def __init__(self, model="bm25", use_stopword=True, use_stemming=True):
        self.korpus = load_corpus().set_index("doc_id")
        self.opsi = dict(use_stopword=use_stopword, use_stemming=use_stemming)
        docs = {d: preprocess(t, **self.opsi) for d, t in corpus_texts().items()}
        self.model = BM25Model(docs) if model == "bm25" else TfidfModel(docs)

    def cari(self, query, k=10):
        return [d for d, _ in self.model.search(preprocess(query, **self.opsi), k=k)]

    def tampilkan(self, query, k=3):
        q_terms = set(preprocess(query, **self.opsi))
        print(f"Hasil pencarian untuk: {query!r}\n")
        for rank, (d, s) in enumerate(self.model.search(list(preprocess(query, **self.opsi)), k=k), 1):
            row = self.korpus.loc[d]
            kalimat = re.split(r"(?<=\.)\s+", row.isi)
            # cuplikan: kalimat dengan term query terbanyak
            cuplikan = max(kalimat, key=lambda s_: len(q_terms & set(preprocess(s_, **self.opsi))))
            print(f"{rank}. [{d}] {row.judul}  (skor {s:.3f})\n   ...{cuplikan}\n")


mesin = MesinPencari()
mesin.tampilkan("cara mencegah diabetes dengan olahraga")

Hasil pencarian untuk: 'cara mencegah diabetes dengan olahraga'

1. [D04] Mengenal Diabetes Melitus  (skor 9.855)
   ...Mengurangi minuman manis dan rajin berolahraga membantu mencegah diabetes.

2. [D03] Manfaat Olahraga bagi Jantung  (skor 4.224)
   ...Olahraga teratur seperti jalan cepat, bersepeda, dan berenang dapat menurunkan risiko penyakit jantung.

3. [D11] Pengendalian Hama Terpadu  (skor 2.509)
   ...Pengendalian hama terpadu menggabungkan cara hayati, mekanis, dan kimia secara bijak.



In [3]:
mesin.tampilkan("sistem pembayaran nontunai untuk pedagang")

Hasil pencarian untuk: 'sistem pembayaran nontunai untuk pedagang'

1. [D21] Pembayaran Digital dengan QRIS  (skor 10.491)
   ...Pedagang kecil dapat menerima pembayaran dari berbagai dompet digital dan aplikasi bank.

2. [D16] Komputasi Awan untuk Bisnis  (skor 2.632)
   ...Layanan komputasi awan dibayar sesuai pemakaian.

3. [D09] Bertanam secara Hidroponik  (skor 2.017)
   ...Sistem ini menghemat penggunaan air dibanding pertanian konvensional.



## Eksperimen ablasi preprocessing
Bagaimana MAP berubah jika stopword removal dan/atau stemming dimatikan?

In [4]:
queries, qrels = load_queries(), load_qrels()
baris = []
for model in ["tfidf", "bm25"]:
    for sw in [False, True]:
        for st in [False, True]:
            m = MesinPencari(model, use_stopword=sw, use_stemming=st)
            run = {q: m.cari(t) for q, t in queries.items()}
            rata = evaluate(run, qrels, k=5).loc["Rata-rata"]
            baris.append({"model": model, "stopword": sw, "stemming": st,
                          "P@5": rata["P@5"], "MAP": rata["AP"], "nDCG@5": rata["nDCG@5"]})
pd.DataFrame(baris)

,model,stopword,stemming,P@5,MAP,nDCG@5
0,tfidf,False,False,0.525,0.7670,0.8894
1,tfidf,False,True,0.500,0.7620,0.8622
2,tfidf,True,False,0.525,0.7682,0.8924
3,tfidf,True,True,0.500,0.7620,0.8622
4,bm25,False,False,0.525,0.8045,0.9036
5,bm25,False,True,0.500,0.7873,0.8720
6,bm25,True,False,0.500,0.7866,0.8861
7,bm25,True,True,0.500,0.7756,0.8668


**Bahan diskusi.** Pada korpus kecil ini stemming justru sedikit menurunkan MAP. Periksa kembali kesalahan stemming yang Anda temukan di Step 1 (misalnya `pemupukan -> pupu` yang tidak cocok dengan `pupuk`, atau `berbasis -> bas`) serta kata umum seperti `ajar` yang menyatukan *pembelajaran daring* dan *pembelajaran mesin*. Apakah kesimpulan yang sama akan berlaku pada korpus yang jauh lebih besar? Rancang eksperimen untuk mengujinya.

## Proyek Akhir Praktikum
Kerjakan secara berkelompok (2–3 orang):

1. **Korpus**: kumpulkan minimal 100 dokumen berbahasa Indonesia dari satu domain (berita, abstrak skripsi, peraturan, resep, dsb.). Cantumkan sumbernya.
2. **Koleksi uji**: buat minimal 10 query dan relevance judgment bertingkat (0/1/2) untuk tiap query.
3. **Sistem**: gunakan kelas `MesinPencari` sebagai dasar. Tambahkan minimal satu peningkatan, misalnya: perbaikan stemming, sinonim/*query expansion*, *pseudo-relevance feedback* (Rocchio), *phrase query*, atau antarmuka web sederhana (Streamlit/Flask).
4. **Evaluasi**: bandingkan sistem dasar dan sistem yang ditingkatkan dengan P@5, MAP, dan nDCG@5.
5. **Laporan**: 6–10 halaman berisi pendahuluan, metode, hasil, analisis kegagalan, dan kesimpulan, disertai notebook yang dapat dijalankan ulang.

In [5]:
# Tulis jawaban Anda di sini